# mlflow.set_tracking_uri

Based on the latest and greatest of **MLflow 3.15.1**.

<br>

```py
def set_tracking_uri(uri: str | Path) -> None
```

Sets the MLflow Tracking Server URI.

The MLflow tracer provider uses tracking URI to determine where to export traces.

In [0]:
%pip install -qU "mlflow-skinny[databricks]"
%restart_python

In [0]:
import mlflow

print(f"MLflow {mlflow.__version__}")

In [0]:
import importlib

print(f"MLflow {importlib.metadata.version("mlflow-skinny")}")


## Pythonic View

* (Re)Imported in `mlflow.__init__.py` from `mlflow.tracking` module.
    * Alongside `get_tracking_uri` and `is_tracking_uri_set`

In [0]:
import mlflow

tracking_uri = mlflow.get_tracking_uri()
print(f"Current tracking uri: {tracking_uri}")

In [0]:
%skip

import mlflow

mlflow.set_tracking_uri("file:///tmp/my_tracking")


# MLFLOW_TRACKING_URI

Sets `MLFLOW_TRACKING_URI` environment variable for subprocesses.

In [0]:
from mlflow.environment_variables import MLFLOW_TRACKING_URI

MLFLOW_TRACKING_URI.get()

In [0]:
from mlflow.utils.databricks_utils import is_in_databricks_model_serving_environment

print(f"is_in_databricks_model_serving_environment: {is_in_databricks_model_serving_environment()}")

In [0]:
from mlflow.tracing.provider import _get_span_processors

print(_get_span_processors())

# BaseMlflowSpanProcessors

There are two `BaseMlflowSpanProcessor`s:

1. `DatabricksUCTableSpanProcessor` for exporting traces to Databricks Unity Catalog table.
1. `MlflowV3SpanProcessor` for exporting traces to MLflow Tracking Server using the V3 trace schema and API.

In [0]:
from mlflow.tracing.provider import _MLFLOW_TRACE_USER_DESTINATION

print(f'destination = {_MLFLOW_TRACE_USER_DESTINATION.get()}')

## Span

A **span** (`opentelemetry.trace.Span`) represents a single operation within a trace.


# Trace Locations

OTel traces are exported to **trace locations** (_destinations_).

Destinations are locations of trace data.

Destinations are subclasses of MLflow's `TraceLocationBase`.

TraceLocationBase | Description | Parameters
-|-|-
 ~~`InferenceTableLocation`~~ | `@deprecated(since="3.7.0")` |
 `MlflowExperimentLocation` | logs OTel traces to an MLflow experiment | `experiment_id`
 `UCSchemaLocation` | a Databricks Unity Catalog (UC) schema for the spans and logs (as fixed, backend-assigned tables).<ul><li>`mlflow_experiment_trace_otel_spans`<li>`mlflow_experiment_trace_otel_logs`</ul>🚨 **WARNING**: Passing `UCSchemaLocation` to `mlflow.tracing.set_destination` is deprecated and will be removed in a future MLflow version.<br><br>Use `set_experiment` with a `UnityCatalog` location instead. Implies `databricks` tracking server URI. See [Store OpenTelemetry traces in Unity Catalog](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog). | `catalog_name` <br> `schema_name`
 `UnityCatalog` | A finer-grained variant of `UCSchemaLocation` that scopes traces under a table prefix inside the schema (`catalog.schema.<prefix>_*`)<br>Separate table names for OTel spans, OTel logs, and annotations.<br>All set by the backend rather than the client.<br><br>📝 **NOTE**: `@experimental(version="3.11.0")` | `catalog_name`<br>`schema_name`<br>`table_prefix` (optional)


In [0]:
from mlflow.entities.trace_location import TraceLocationType

for tt in TraceLocationType:
    print(tt)

## TraceDestinations Deprecated

Previously, `TraceLocationBase`s were defined as `TraceDestination`s in `mlflow.tracing.destination` module.

They were `@deprecated(since="3.5.0")` for `mlflow.entities.trace_location` locations above.

# UnityCatalog Trace Location

`@experimental(version="3.11.0")`

`UnityCatalog` is a `TraceLocationBase` that represents a Databricks Unity Catalog location with a table prefix.

Scopes traces under a table prefix inside the schema (`catalog.schema.<prefix>_*`), with separate table names for OTel spans, OTel logs, and annotations.

In [0]:
%skip

import mlflow
from mlflow.entities.trace_location import UnityCatalog

mlflow.set_experiment(
    experiment_id="experiment_id",
    trace_location=UnityCatalog(
        catalog_name="jacek_laskowski",
        schema_name="traces",
        table_prefix="my_prefix",
    ),
)

## How it works

- `mlflow.set_experiment(..., trace_location=UnityCatalog(...))` (`mlflow/tracking/fluent.py:145`) links an experiment to a UC destination — if `table_prefix` isn't given, it defaults to the experiment ID.
- The `MLFLOW_TRACING_DESTINATION` env var can also point at `<catalog>.<schema>` for a `UCSchemaLocation`, but explicitly rejects a 3-part catalog.schema.table_prefix
string, directing users to the `UnityCatalog` API instead (`mlflow/tracing/destination.py`).
- At trace-export time, `DatabricksUCTableSpanProcessor._start_trace` (`mlflow/tracing/processor/uc_table.py`) checks the active `_MLFLOW_TRACE_USER_DESTINATION`: if it's a UnityCatalog, spans go to the table-prefix-scoped tables; if `UCSchemaLocation`, to the schema-level default tables.
- `DatabricksUCTableSpanExporter` (`mlflow/tracing/export/uc_table.py`) then does the actual write via `client.log_spans(location, spans)`, batched asynchronously when
`MLFLOW_ENABLE_ASYNC_TRACE_LOGGING` is set.

Thanks Claude 👏👏👏


## Interesting findings

In `mlflow/utils/rest_utils.py`:

1. `_UC_OSS_REST_API_PATH_PREFIX = "/api/2.1"`
1. `_V4_REST_API_PATH_PREFIX = "/api/4.0"`
1. `_V4_TRACE_REST_API_PATH_PREFIX = f"{_V4_REST_API_PATH_PREFIX}/mlflow/traces"`


## Use Cases

1. **Storing GenAI/agent traces directly in Unity Catalog** instead of (or alongside) MLflow's own tracking backend — useful when traces need to live alongside other
governed data in the Databricks lakehouse for downstream SQL/analytics access.
2. **Schema-wide trace routing** (`UCSchemaLocation`) — simplest setup, all traces for a schema go to one fixed pair of OTel tables; good for teams that just want "traces go to this UC schema."
3. **Per-experiment/per-app table-prefix isolation** (`UnityCatalog` with `table_prefix`) — lets multiple experiments/apps share a UC schema while keeping their
spans/logs/annotations in separate, prefixed tables (defaulting the prefix to the experiment ID is a deliberate convenience for the common "one experiment = one app"
case).
4. **Governed, queryable trace storage for observability** — since spans/logs land in real UC tables (not opaque artifact blobs), they become queryable via SQL/Databricks tooling for monitoring, debugging, and building dashboards over agent behavior.

## Upsell Path

There's also an "upsell" path (`_MLFLOW_ENABLE_UC_TRACE_UPSELL`, `show_new_experiment_upsell`/`show_existing_experiment_upsell`) nudging Databricks users toward UC trace destinations when an experiment has no trace destination tag set yet — suggesting this is a strategic/product direction, not just a niche feature.

From [Migrate experiment traces to Unity Catalog](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc):

* Databricks recommends storing MLflow traces in Unity Catalog delta tables for new and production workloads.
    * No trace storage limits.
    * Fine-grained access control (Unity Catalog governance).
    * Traces queryable from notebooks, SQL, Genie, AI/BI dashboards, and any Spark-based tool.
* Logs traces, spans, assessments, tags, and metadata from the source experiment to Unity Catalog tables.
* Stored in OpenTelemetry (OTel) format.
* Existing traces stored in an MLflow experiment should be migrated to Unity Catalog.
* Requires MLflow 3.14 or later.

Learn more in [Store OpenTelemetry traces in Unity Catalog](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog).

![Migrate your traces to Unity Catalog Info Message in Databricks UI](./databricks_migrate_traces_to_unity_catalog.png)

# 👨‍💻 Demo

In [0]:
import mlflow

from mlflow.entities.trace_location import UnityCatalog

In [0]:
assert mlflow.get_tracking_uri() == "databricks"

In [0]:
# Specify the catalog, schema, and table prefix to use for storing Traces
catalog_name = "jacek_laskowski_v2"
schema_name = "mlflow"
table_prefix = "demo_v1"

In [0]:
sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

In [0]:
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
print(f"Notebook path: {notebook_path}")

In [0]:
%skip

# It does not seem necessary
# The required OTel tables will be created automatically

# https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog#grant-permissions
# MODIFY and SELECT on each of the <table_prefix>_<type> tables.
# ALL_PRIVILEGES is not sufficient for accessing Unity Catalog trace tables. You must explicitly grant MODIFY and SELECT.

types = ["spans", "logs", "annotations", "assessments", "tags", "metadata"]
user_or_group = spark.sql("SELECT current_user()").collect()[0][0]

print(f"Granting MODIFY and SELECT on {types} tables to {user_or_group!r}")

for t in types:
    table_name = f"{catalog_name}.{schema_name}.{table_prefix}_{t}"
    sql(f"CREATE TABLE IF NOT EXISTS {table_name} USING DELTA")
    sql(f"GRANT MODIFY, SELECT ON TABLE {table_name} TO `{user_or_group}`")

In [0]:
# In Databricks, the experiment name must be an absolute path (e.g. "/Users/<username>/my-experiment").
experiment_name=notebook_path

# For existing experiments, it is not necessary to specify `trace_location`.
# MLflow retrieves the UC trace location bound to the experiment and routes traces to that location.
# FIXME: How?!

mlflow.set_experiment(
    experiment_name=experiment_name,
    trace_location=UnityCatalog(
        catalog_name=catalog_name,
        schema_name=schema_name,
        table_prefix=table_prefix,
    ),
)

## ⚠️ Unsupported table kind

OTel tables created in default storage are not supported (and the code fails with "Error Code: 4024, Error State: 3.")

* `metastore_aws_us_west_2`
* **Storage configuration name**: `s3_bucket`
* **S3 bucket path**: `s3://japila-uc-mlflow-traces/otel-traces`
* **IAM role ARN**: `arn:aws:iam::212268664488:role/databricks_unity_catalog`

In [0]:
import mlflow

# Create and ingest an example trace using the `@mlflow.trace` decorator
@mlflow.trace
def test(x):
    return x + 1

test(100)